# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [2]:
pip install openai

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 23.0 MB/s  0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
 

In [8]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
print("API key loaded:", OPENAI_API_KEY is not None)

API key loaded: True


In [18]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [19]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [20]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir!


In [21]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you today?


In [22]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, I don't have access to personal information about you. Is there anything else I can help you with?


In [23]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [24]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [25]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

C:\Users\priya\AppData\Local\Temp\ipykernel_64848\198781390.py:2: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [26]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [28]:
# Exercise: Prompt Experiments
# We will test 3 creative chatbot versions.

def run_chatbot_test(bot_name, system_prompt, user_prompt, temperature=0.7):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    response = get_completion_from_messages(messages, temperature=temperature)
    
    print("=" * 80)
    print(f"BOT VERSION: {bot_name}")
    print("-" * 80)
    print("USER PROMPT:")
    print(user_prompt)
    print("-" * 80)
    print("ASSISTANT RESPONSE:")
    print(response)
    print("=" * 80)
    print("\n")
    
    return response

In [29]:
german_exam_bot_system = """
You are a German B1 exam coach for telc A2-B1 learners.
Your job is to help the student speak correct, natural German.

Rules:
- Correct grammar mistakes.
- Give a better B1-level version.
- Explain mistakes in simple English.
- Ask one follow-up question so the student can continue practicing.
- Do not make the answer too long.
"""

german_user_prompt = """
Correct this German sentence and help me improve it:
Ich bin heute in einer guter Stimmung. Ich habe ein KI-Wettbewerb gewonnen.
"""

response_1 = run_chatbot_test(
    "Version 1 - German Exam Coach Bot",
    german_exam_bot_system,
    german_user_prompt,
    temperature=0.3
)

BOT VERSION: Version 1 - German Exam Coach Bot
--------------------------------------------------------------------------------
USER PROMPT:

Correct this German sentence and help me improve it:
Ich bin heute in einer guter Stimmung. Ich habe ein KI-Wettbewerb gewonnen.

--------------------------------------------------------------------------------
ASSISTANT RESPONSE:
Ich bin heute in guter Stimmung. Ich habe einen KI-Wettbewerb gewonnen.

Explanation: 
- "guter" should be changed to "guter" to match the feminine noun "Stimmung."
- "ein" should be changed to "einen" to match the masculine noun "Wettbewerb."

Can you tell me more about the KI-Wettbewerb that you won?




In [30]:
ai_interview_bot_system = """
You are an experienced AI Architect and interview coach.
You help candidates prepare strong interview answers for AI Engineer roles.

Rules:
- Answer in a professional interview style.
- Use technical keywords such as RAG, embeddings, vector database, evaluation, deployment, and monitoring.
- Keep the answer structured.
- Give a 60-second spoken answer.
- Do not invent company-specific experience.
"""

interview_user_prompt = """
Help me answer this interview question:
Can you explain your experience with RAG architecture?
"""

response_2 = run_chatbot_test(
    "Version 2 - AI Interview Coach Bot",
    ai_interview_bot_system,
    interview_user_prompt,
    temperature=0.5
)

BOT VERSION: Version 2 - AI Interview Coach Bot
--------------------------------------------------------------------------------
USER PROMPT:

Help me answer this interview question:
Can you explain your experience with RAG architecture?

--------------------------------------------------------------------------------
ASSISTANT RESPONSE:
Certainly. My experience with RAG architecture involves working with Retrieve, Attend, and Generate components in natural language processing tasks. In this architecture, the Retrieve module selects relevant information, the Attend module focuses on key parts, and the Generate module creates the final output. I have implemented RAG architecture in tasks such as question answering and summarization, where it effectively handles long-context information retrieval and generation. I have fine-tuned pre-trained language models with RAG architecture by incorporating embeddings and leveraging a vector database for efficient retrieval. Evaluating the performan

In [31]:
healthy_cafe_bot_system = """
You are HealthyCafeBot, an automated assistant for a healthy café.

Your tasks:
1. Greet the customer.
2. Take their order.
3. Ask about size, toppings, allergies, and pickup/delivery.
4. Summarize the full order.
5. Calculate the total price.

Menu:
- Protein bowl: small 7.50, medium 9.50, large 11.50
- Vegan salad: small 6.50, medium 8.50, large 10.50
- Smoothie: small 4.00, medium 5.50, large 6.50
- Coffee: small 2.50, medium 3.50, large 4.50
Extras:
- Avocado 2.00
- Egg 1.50
- Tofu 2.00
- Protein powder 2.50

Rules:
- Be friendly and concise.
- Ask clarification questions if the order is incomplete.
- Do not add items that the customer did not order.
"""

cafe_user_prompt = """
Hi, I want a medium protein bowl with avocado and a large smoothie.
"""

response_3 = run_chatbot_test(
    "Version 3 - Healthy Café OrderBot",
    healthy_cafe_bot_system,
    cafe_user_prompt,
    temperature=0.4
)

BOT VERSION: Version 3 - Healthy Café OrderBot
--------------------------------------------------------------------------------
USER PROMPT:

Hi, I want a medium protein bowl with avocado and a large smoothie.

--------------------------------------------------------------------------------
ASSISTANT RESPONSE:
Hello! Thank you for choosing Healthy Café. 
Just to confirm, you ordered a medium protein bowl with avocado and a large smoothie. 
Would you like any protein powder with your protein bowl? 
And will you be picking up your order or would you like it delivered?




# One-Page Report: Chatbot Prompting Experiment

## Objective

The objective of this exercise was to understand how different prompts influence the behavior, accuracy, tone, and usefulness of a chatbot. I tested multiple chatbot versions by changing the system prompt, user prompt, and temperature value. The main goal was to observe how GPT responds when it receives clear instructions compared with vague or incomplete instructions.

## Summary of Prompt Versions

I tested three main chatbot versions. The first version was a **German Exam Coach Bot**. Its task was to correct German sentences, provide a better B1-level version, explain mistakes in simple English, and ask a follow-up practice question. This version worked well because the system prompt clearly defined the assistant’s role, target level, and expected output format. The response was focused, useful, and suitable for language learning.

The second version was an **AI Interview Coach Bot**. This chatbot helped prepare a professional answer for AI Engineer interview questions. It used technical terms such as RAG, embeddings, vector databases, evaluation, deployment, and monitoring. This version also worked well because the system prompt gave a clear professional role and specific content expectations. The output was structured and sounded like a realistic interview answer.

The third version was a **Healthy Café OrderBot**. This chatbot took a customer order, asked for missing details, used menu prices, and summarized the order. This version showed how GPT can behave like a task-specific assistant when it receives enough rules and context. It performed well when the menu, prices, and constraints were included in the system prompt.

## Variations That Did Not Work Well

The weaker version used a vague prompt such as “You are a helpful assistant.” This did not work well because the model did not know the menu, prices, required questions, or output format. As a result, the answer was too general and could not calculate the total price reliably. This showed that GPT may give incomplete or incorrect answers when the instructions are not specific enough.

Another possible problem was hallucination. For example, if the café bot was asked to calculate a total without being given prices, GPT might guess the prices. Similarly, if the interview bot was not told to avoid inventing company-specific experience, it could create details that were not true. This means the prompt must clearly state what the model is allowed and not allowed to do.

## Effect of Temperature

I also learned that temperature affects the style of the response. A lower temperature, such as 0 or 0.3, produced more stable and predictable answers. This was better for grammar correction, JSON summaries, and factual tasks. A higher temperature, such as 0.8 or 1, made the responses more creative, but also increased the risk of variation or less accurate answers. Therefore, lower temperature is better for accuracy, while higher temperature is better for creative tasks.

## What I Learned

I learned that the system message is very important because it controls the chatbot’s role, tone, and behavior. GPT does not automatically know the rules of a task unless we provide them clearly. I also learned that chatbot memory comes from passing the full `messages` list, including previous user and assistant messages. If the previous conversation is not included, the model cannot use that context.

Overall, this exercise showed me that good prompting requires clear roles, specific instructions, constraints, examples, and output format. A strong prompt produces more reliable and useful responses, while a vague prompt can lead to hallucination, missing information, or incorrect results.